# KB Previl — 검증 노트북

이 노트북은 위에서부터 차례로 실행하면서 **서비스가 주장하는 것이 실제로 그런지**를
확인합니다. 어긋나는 것이 있으면 그 칸에서 멈춥니다.

| | 확인하는 것 |
|---|---|
| ① | 데이터가 다 들어와 있는가 |
| ② | 등급이 실제로 생존율 순인가 |
| ③ | 추천 API 가 등급 순으로 답하는가 |
| ④ | 자리 상세가 근거를 갖춰 나오는가 |
| ⑤ | 화면이 서빙되는가 |

실행 전에 `python run.py` 를 한 번 돌린 뒤, `requirements-full.txt` 를
설치해야 합니다(그래프·노트북 실행 라이브러리 — README «검증 노트북» 절의
두 줄 그대로).

In [ ]:
import json
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "kb-demo.db").is_file() and (ROOT.parent / "kb-demo.db").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DB = ROOT / "kb-demo.db"
assert DB.is_file(), f"kb-demo.db 를 찾지 못했습니다. 이 노트북과 같은 폴더에 두세요: {ROOT}"

con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
BLUE, GREY = "#3b6fb0", "#b8c0cc"

print("폴더 :", ROOT)
print("DB   :", DB.name, f"({DB.stat().st_size / 1024**2:,.0f} MB)")
print("준비 완료")

---
## ① 데이터가 다 들어와 있는가

서빙이 실제로 읽는 표만 셉니다. 하나라도 비어 있으면 화면 어딘가가 조용히 빈
채로 나가기 때문에, 여기서 멈추는 편이 낫습니다.

In [ ]:
TABLES = ["licence", "licence_rest", "grid", "grid_feature", "grid_score",
          "succession_score", "grid_concept", "grid_access", "grid_sgis",
          "trdar_sales", "trdar_store", "trdar_party", "cohort_survival",
          "score_meta"]

counts = {t: con.execute(f'select count(*) from "{t}"').fetchone()[0] for t in TABLES}
for t, n in counts.items():
    print(f"  {t:<18} {n:>10,}")

empty = [t for t, n in counts.items() if n == 0]
assert not empty, f"비어 있는 표가 있습니다: {empty}"
print(f"\n[PASS] 표 {len(TABLES)}종 모두 채워져 있습니다.")

fig, ax = plt.subplots()
names = list(counts)[::-1]
ax.barh(names, [counts[t] for t in names], color=BLUE, height=0.62)
ax.set_xscale("log")
ax.set_xlabel("rows (log scale)")
ax.set_title("Serving tables")
for i, t in enumerate(names):
    ax.text(counts[t] * 1.15, i, f"{counts[t]:,}", va="center", fontsize=8)
ax.set_xlim(right=max(counts.values()) * 6)
plt.tight_layout(); plt.show()

---
## ② 등급이 실제로 생존율 순인가

**이 서비스의 핵심 주장입니다.** 2022년까지의 기록만으로 배운 모델이 매긴 등급이,
한 번도 보지 못한 **2023년 개업 가게들**의 실제 3년 생존율과 순서가 맞는지 봅니다.

등급이 낮아질수록 생존율이 **한 번도 뒤집히지 않고** 떨어져야 합니다.

In [ ]:
meta = dict(con.execute("select k, v from score_meta"))
point = [float(x) for x in meta["observed_by_grade"].split(",")]
lo = [float(x) for x in meta["observed_by_grade_ci_low"].split(",")]
hi = [float(x) for x in meta["observed_by_grade_ci_high"].split(",")]
n = [int(x) for x in meta["observed_by_grade_n"].split(",")]
grades = list(range(1, len(point) + 1))

assert len(point) == len(lo) == len(hi) == len(n) == 9, "등급이 9개가 아닙니다"
for i in range(8):
    assert point[i] > point[i + 1], f"{i + 1}등급이 {i + 2}등급보다 낮습니다 — 순서가 뒤집혔습니다"
for i in range(9):
    assert lo[i] <= point[i] <= hi[i], f"{i + 1}등급 신뢰구간이 점추정을 담지 못합니다"

print("  등급   생존율    95% 신뢰구간        표본")
for g, p, a, b, k in zip(grades, point, lo, hi, n):
    print(f"   {g}    {p * 100:5.1f}%   {a * 100:5.1f} ~ {b * 100:5.1f}%   {k:>6,}곳")
print(f"\n  양끝 격차 {(point[0] - point[-1]) * 100:.1f}%p · 표본 합계 {sum(n):,}곳")
print("\n[PASS] 9개 등급이 한 번도 뒤집히지 않고 생존율 순입니다.")

fig, ax = plt.subplots()
err = [[p - a for p, a in zip(point, lo)], [b - p for p, b in zip(point, hi)]]
ax.errorbar(grades, [p * 100 for p in point],
            yerr=[[e * 100 for e in err[0]], [e * 100 for e in err[1]]],
            fmt="o-", color=BLUE, ecolor=GREY, elinewidth=2, capsize=5,
            markersize=7, linewidth=2)
ax.set_xticks(grades)
ax.set_xlabel("Location grade (1 = best)")
ax.set_ylabel("3-year survival (%)")
ax.set_title("Observed survival by grade — 2023 cohort, held out")
ax.set_ylim(0, 100)
for g, p, k in zip(grades, point, n):
    ax.annotate(f"{p * 100:.1f}%", (g, p * 100), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=8)
plt.tight_layout(); plt.show()

---
## ③ 추천 API 가 등급 순으로 답하는가

실제 서버를 띄우지 않고 앱을 직접 불러 확인합니다. 추천 목록은 **좋은 자리부터**
나와야 하고, 각 후보는 자기 등급과 실측 생존율을 달고 나와야 합니다.

In [ ]:
from fastapi.testclient import TestClient
from service.app import app

client = TestClient(app)

m = client.get("/api/meta")
assert m.status_code == 200, f"/api/meta 가 {m.status_code} 를 냈습니다"
meta_json = m.json()
uptae = meta_json["uptae"][0]
print(f"업종 {len(meta_json['uptae'])}종 · 자치구 {len(meta_json['districts'])}개 "
      f"· 격자 {meta_json['gridCount']:,}칸 · 기준 {meta_json['asOf']}")

r = client.get("/api/recommend", params={"uptae": uptae, "limit": 10})
assert r.status_code == 200, f"/api/recommend 가 {r.status_code} 를 냈습니다"
items = r.json()["items"]
assert items, "추천 결과가 비어 있습니다"

surv = [it["observedSurvival"] for it in items]
assert surv == sorted(surv, reverse=True), "추천이 생존율 내림차순이 아닙니다"
grade_seq = [it["grade"] for it in items]
assert grade_seq == sorted(grade_seq), "등급이 오름차순이 아닙니다"

print(f"\n«{uptae}» 추천 상위 {len(items)}곳")
print("  등급  생존율   자치구      행정동         가장 가까운 역")
for it in items:
    st = it.get("nearestStation") or {}
    print(f"   {it['grade']}   {it['observedSurvival'] * 100:5.1f}%  "
          f"{it['district']:<8} {it['admDong']:<12} "
          f"{st.get('name', '—'):<8} {st.get('distanceM', '')}")
print("\n[PASS] 추천이 좋은 자리부터 순서대로 나옵니다.")

fig, ax = plt.subplots()
labels = [f"{it['admDong']}\n{it['gridId']}" for it in items]
ax.bar(range(len(items)), [s * 100 for s in surv], color=BLUE, width=0.62)
ax.set_xticks(range(len(items)))
ax.set_xticklabels([f"#{i + 1}" for i in range(len(items))])
ax.set_ylabel("3-year survival (%)")
ax.set_title(f"Top {len(items)} recommendations — {uptae}")
ax.set_ylim(0, 100)
for i, (s, it) in enumerate(zip(surv, items)):
    ax.text(i, s * 100 + 2, f"grade {it['grade']}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

---
## ④ 자리 상세가 근거를 갖춰 나오는가

추천 1위 자리를 열어, 화면의 카드들이 실제로 값을 갖고 나오는지 봅니다.
**없는 것은 지어내지 않고 비운다**는 규칙도 함께 확인합니다 — 상권 밖 자리는
매출 기반 항목이 `available: false` 로 나와야 합니다.

In [ ]:
top = items[0]
d = client.get(f"/api/grid/{top['gridId']}", params={"uptae": uptae})
assert d.status_code == 200, f"/api/grid 가 {d.status_code} 를 냈습니다"
detail = d.json()

REQUIRED = ["gridId", "grade", "observedSurvival", "polygon", "center",
            "competition", "conceptMix", "visitorParty", "salesMix",
            "areaSurvival", "missingAxes", "resolutions"]
missing = [k for k in REQUIRED if k not in detail]
assert not missing, f"상세 응답에 빠진 항목: {missing}"
assert len(detail["polygon"]) >= 4, "격자 폴리곤이 사각형이 아닙니다"

print(f"{detail['district']} {detail['admDong']} · {detail['gridId']}")
print(f"  {detail['grade']}등급 · 실측 생존율 {detail['observedSurvival'] * 100:.1f}%")
print(f"  상권 매출 자료 {'있음' if detail['salesAvailable'] else '없음 (상권 밖)'}")
print()
for key in ("competition", "conceptMix", "visitorParty", "salesMix"):
    block = detail.get(key) or {}
    if isinstance(block, dict) and "available" in block:
        state = "값 있음" if block.get("available") else "비어 있음 (자료 없음)"
        size = len(block.get("items") or [])
        print(f"  {key:<14} {state:<22} 항목 {size}개")
    else:
        print(f"  {key:<14} {'값 있음' if block else '비어 있음'}")

for key in ("visitorParty", "salesMix", "conceptMix"):
    block = detail.get(key) or {}
    if block.get("available"):
        assert block.get("items"), f"{key} 가 available 인데 내용이 비었습니다"
print("\n[PASS] 상세가 근거를 갖춰 나오고, 없는 자료는 비운 채로 표시됩니다.")

mix = (detail.get("salesMix") or {}).get("items") or []
if mix:
    fig, ax = plt.subplots(figsize=(9, 3.6))
    mix = mix[:8][::-1]
    ax.barh([m["induty"] for m in mix], [m["share"] * 100 for m in mix],
            color=BLUE, height=0.62)
    ax.set_xlabel("share of card payments (%)")
    ax.set_title("Where money is spent in this trade area")
    for i, m in enumerate(mix):
        ax.text(m["share"] * 100 + 0.6, i, f"{m['share'] * 100:.1f}%",
                va="center", fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("(이 자리는 상권 밖이라 매출 구성 그래프가 없습니다)")

---
## ⑤ 화면이 서빙되는가

빌드된 화면이 같은 서버에서 나가는지, 지도가 쓰는 모듈이 올바른 타입으로
서빙되는지 확인합니다. 지도 모듈이 `text/plain` 으로 나가면 브라우저가 실행을
거부해서 **지도에 배경만 뜨고 등급 칸이 하나도 안 그려집니다.**

In [ ]:
import re

ui = client.get("/")
assert ui.status_code == 200, f"화면이 {ui.status_code} 를 냈습니다"
assert b"<div id=\"root\"" in ui.content or b"<script" in ui.content, \
    "index.html 이 앱 진입점을 담고 있지 않습니다"
print(f"화면        {ui.status_code} · {len(ui.content):,} bytes · "
      f"{ui.headers.get('content-type')}")

web = ROOT / "web"
if not web.is_dir():
    web = ROOT / "frontend" / "app" / "dist"
wanted = set()
for js in (web / "assets").glob("*.js"):
    wanted |= set(re.findall(r'"([\w.-]+\.mjs)"', js.read_text("utf-8", "ignore")))

seen, bad = set(), []
while wanted:
    name = wanted.pop()
    if name in seen or name.endswith("-dev.mjs"):
        continue
    seen.add(name)
    resp = client.get(f"/assets/{name}")
    ctype = (resp.headers.get("content-type") or "").split(";")[0]
    if resp.status_code != 200:
        bad.append(f"{name} → {resp.status_code}")
    elif "javascript" not in ctype and "ecmascript" not in ctype:
        bad.append(f"{name} → {ctype}")
    else:
        wanted |= set(re.findall(r'from\s*["\']\./([\w.-]+\.mjs)["\']', resp.text))
        print(f"지도 모듈    {name}  {resp.status_code} · {ctype}")

assert seen, "지도 모듈을 하나도 찾지 못했습니다 — 화면 빌드가 비어 있습니다"
assert not bad, f"잘못 서빙되는 모듈: {bad}"

g = client.get("/api/grids", params={"uptae": uptae,
                                     "bbox": "127.024,37.494,127.032,37.501"})
assert g.status_code == 200, f"/api/grids 가 {g.status_code} 를 냈습니다"
cells_on_map = len(g.json().get("items") or [])
assert cells_on_map > 0, "지도에 그릴 격자가 없습니다"
print(f"지도 데이터  {g.status_code} · 격자 {cells_on_map}칸")
print("\n[PASS] 화면과 지도가 정상 서빙됩니다.")

---
## 마무리

다섯 가지가 모두 통과했다면, 이 서비스는

- **데이터가 다 들어와 있고**
- **등급이 실제 생존율과 순서가 맞고**
- **추천·상세·화면이 모두 응답합니다.**

한 가지만 덧붙입니다. **1등급 자리에서도 5곳 중 1곳은 3년 안에 문을 닫습니다.**
위 ② 그래프의 신뢰구간이 그 폭을 그대로 보여 줍니다. 등급은 확률이지 보장이
아닙니다.